# ST-OMR Meter V5-2A — 300 TRAIN Full-Meter BBox

Bu notebook yalnız TRAIN annotation içindir. İlk 30 V5-1 BBox kilitlidir; 270 yeni full-meter BBox çizilir. TRAINING=CLOSED, VAL=CLOSED, FINAL_HOLDOUT=LOCKED, 4-AI=FROZEN CONTROL.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('DRIVE_MOUNT=PASS')


In [ ]:
import os, shutil, subprocess, sys, threading, time
from IPython.display import clear_output, display, HTML

CODE_SHA = '04e86c57a719aa84714ee476d64c19dff72bee8c'
REPO = '/content/st-omr-training-v52a'

def run_monitored_background(title, worker, heartbeat=5):
    state = {'phase':'starting','detail':'','error':None,'result':None}
    def wrapped():
        try:
            state['result'] = worker(state)
            state['phase'] = 'done'
        except BaseException as exc:
            state['error'] = exc
            state['phase'] = 'error'
    thread = threading.Thread(target=wrapped, daemon=False, name=title)
    started = time.monotonic()
    thread.start()
    while thread.is_alive():
        clear_output(wait=True)
        display(HTML(f'''<div style="border:2px solid #555;border-radius:10px;padding:12px;max-width:950px"><b>{title}</b><br><b>Durum:</b> RUNNING<br><b>Faz:</b> {state['phase']}<br><b>Geçen süre:</b> {int(time.monotonic()-started)} s<br><b>Detay:</b> {state['detail']}<br><br><b>Güvenlik:</b> TRAINING=CLOSED | VAL=CLOSED | FINAL_HOLDOUT=LOCKED | 4-AI=FROZEN</div>'''))
        thread.join(timeout=heartbeat)
    thread.join()
    clear_output(wait=True)
    status = 'FAIL-CLOSED' if state['error'] else 'PASS'
    display(HTML(f'''<div style="border:2px solid #555;border-radius:10px;padding:12px;max-width:950px"><b>{title}</b><br><b>Durum:</b> {status}<br><b>Faz:</b> {state['phase']}<br><b>Geçen süre:</b> {int(time.monotonic()-started)} s<br><b>Detay:</b> {state['detail']}<br><br><b>Güvenlik:</b> TRAINING=CLOSED | VAL=CLOSED | FINAL_HOLDOUT=LOCKED | 4-AI=FROZEN</div>'''))
    if state['error']:
        raise state['error']
    return state['result']

def setup_worker(state):
    state['phase'] = 'clone'
    state['detail'] = 'clean clone + exact SHA checkout'
    shutil.rmtree(REPO, ignore_errors=True)
    subprocess.run(['git','clone','-q','https://github.com/khfy7wpr5p-maker/st-omr-training.git',REPO], check=True)
    subprocess.run(['git','-C',REPO,'checkout','-q',CODE_SHA], check=True)
    state['phase'] = 'dependency'
    state['detail'] = 'Pillow 12.3.0'
    subprocess.run([sys.executable,'-m','pip','install','-q','Pillow==12.3.0'], check=True)
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    state['detail'] = 'exact code SHA ready'
    return True

run_monitored_background('V5-2A SETUP İZLEME', setup_worker, heartbeat=5)
print('CODE_SHA=', CODE_SHA)


In [ ]:
from pathlib import Path
from st_omr_training.meter_v5_2a_specialist_adaptation import AdaptationAnnotationSession, TRAIN_TOTAL

DATA_ROOT = Path('/content/drive/MyDrive/TEST/METER_V2_1500_PACKAGE_AB_CLEAN')
if not DATA_ROOT.is_dir():
    raise RuntimeError(f'Dataset root missing: {DATA_ROOT}')

def precheck_worker(state):
    state['phase'] = 'seed + dataset verification'
    state['detail'] = 'exact V5-1 seed hashes + deterministic 300 TRAIN binding'
    session = AdaptationAnnotationSession(data_root=DATA_ROOT)
    if len(session.samples) != 300 or TRAIN_TOTAL != 300:
        raise RuntimeError('V5-2A selection is not exactly 300')
    if session.handled_count < 30 or session.pass_count < 30 or session.resume_index() < 30:
        raise RuntimeError('V5-1 30-seed precondition failed')
    state['detail'] = f'handled={session.handled_count} pass={session.pass_count} review={session.review_count} resume={session.resume_index()}'
    return session

SESSION_300 = run_monitored_background('V5-2A PRECHECK İZLEME', precheck_worker, heartbeat=5)
print('PRECHECK=PASS')
print('handled=', SESSION_300.handled_count, 'pass=', SESSION_300.pass_count, 'review=', SESSION_300.review_count)
print('resume_index=', SESSION_300.resume_index())
print('TARGET=300 | SEEDS=30_LOCKED | NEW_REMAINING=', 300-SESSION_300.handled_count)
print('TRAINING=False | VAL=False | FINAL_HOLDOUT=LOCKED | MODEL_OPENED=False')


In [ ]:
from st_omr_training.meter_v5_2a_annotation_colab import launch_colab_annotation
launch_colab_annotation(data_root=str(DATA_ROOT), session=SESSION_300)


In [ ]:
from st_omr_training.meter_v5_2a_specialist_adaptation import write_annotation_audit

def audit_worker(state):
    state['phase'] = 'mechanical audit'
    state['detail'] = '300 TRAIN bindings + class balance + seed immutability'
    path = write_annotation_audit(DATA_ROOT)
    state['detail'] = str(path)
    return path

AUDIT_PATH = run_monitored_background('V5-2A AUDIT İZLEME', audit_worker, heartbeat=5)
print(AUDIT_PATH.read_text(encoding='utf-8'))
print('NOTE: mechanical PASS still does NOT authorize training; human visual QA/contact-sheet review is required.')
